# CeKALA Experiment Notebook
This notebook allows you to easily run the Multi-Modal Adapter (MMA) experiments on Google Colab.

In [2]:
# 2. Git setup and repository update
repo_url = "https://github.com/tomal66/CeKALA.git"
repo_dir = "CeKALA"
branch = "main"

if not os.path.exists(repo_dir):
    !git clone {repo_url}
    %cd {repo_dir}
    !git checkout {branch}
else:
    %cd {repo_dir}
    !git fetch origin {branch}
    !git reset --hard origin/{branch}

!pip install -r requirements.txt

/content/CeKALA
From https://github.com/tomal66/CeKALA
 * branch            main       -> FETCH_HEAD
HEAD is now at 7e3da3e Initial commit
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/dassl-0.6.3-py3.12.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330


In [3]:
# 3. Hyperparameters
DATASET = "caltech101"
SEED = 1
CFG = "vit_b16_ep5"
SHOTS = 16
DATA = "data/"
TRAINER = "MultiModalAdapter"
LOADEP = 5
SUB = "new"

In [4]:
# 4. Imports
import sys
import torch

from dassl.utils import setup_logger, set_random_seed, collect_env_info
from dassl.engine import build_trainer

# Custom modules from CeKALA
import datasets.oxford_pets
import datasets.oxford_flowers
import datasets.fgvc_aircraft
import datasets.dtd
import datasets.eurosat
import datasets.stanford_cars
import datasets.food101
import datasets.sun397
import datasets.caltech101
import datasets.ucf101
import datasets.imagenet

import datasets.imagenet_sketch
import datasets.imagenetv2
import datasets.imagenet_a
import datasets.imagenet_r

import trainers.mmadapter
from train import setup_cfg, print_args

ModuleNotFoundError: No module named 'datasets.oxford_pets'

In [ ]:
# 5. Setup Arguments and Config for Base Training
class Args:
    pass

args_train = Args()
args_train.root = DATA
args_train.output_dir = f"output/base2new/train_base/{DATASET}/shots_{SHOTS}/{TRAINER}/seed{SEED}"
args_train.resume = ""
args_train.seed = SEED
args_train.source_domains = None
args_train.target_domains = None
args_train.transforms = None
args_train.config_file = f"configs/trainers/{TRAINER}/{CFG}.yaml"
args_train.dataset_config_file = f"configs/datasets/{DATASET}.yaml"
args_train.trainer = TRAINER
args_train.backbone = ""
args_train.head = ""
args_train.eval_only = False
args_train.model_dir = ""
args_train.load_epoch = None
args_train.no_train = False
args_train.opts = ["DATASET.NUM_SHOTS", str(SHOTS), "DATASET.SUBSAMPLE_CLASSES", "base"]

cfg_train = setup_cfg(args_train)
if cfg_train.SEED >= 0:
    print(f"Setting fixed seed: {cfg_train.SEED}")
    set_random_seed(cfg_train.SEED)

setup_logger(args_train.output_dir)

if torch.cuda.is_available() and cfg_train.USE_CUDA:
    torch.backends.cudnn.benchmark = True

print_args(args_train, cfg_train)
print("Collecting env info ...")
print("** System info **\n{}\n".format(collect_env_info()))

print("Building trainer...")
trainer = build_trainer(cfg_train)

In [ ]:
# 6. Start Training on Base Classes
print("Starting training on base classes...")
trainer.train()

In [ ]:
# 7. Setup Arguments and Config for New Classes Testing
args_test = Args()
args_test.root = DATA
args_test.output_dir = f"output/base2new/test_{SUB}/{DATASET}/shots_{SHOTS}/{TRAINER}/seed{SEED}"
args_test.resume = ""
args_test.seed = SEED
args_test.source_domains = None
args_test.target_domains = None
args_test.transforms = None
args_test.config_file = f"configs/trainers/{TRAINER}/{CFG}.yaml"
args_test.dataset_config_file = f"configs/datasets/{DATASET}.yaml"
args_test.trainer = TRAINER
args_test.backbone = ""
args_test.head = ""
args_test.eval_only = True
args_test.model_dir = args_train.output_dir
args_test.load_epoch = LOADEP
args_test.no_train = True
args_test.opts = ["DATASET.NUM_SHOTS", str(SHOTS), "DATASET.SUBSAMPLE_CLASSES", SUB]

cfg_test = setup_cfg(args_test)
if cfg_test.SEED >= 0:
    set_random_seed(cfg_test.SEED)
setup_logger(args_test.output_dir)

trainer_test = build_trainer(cfg_test)
trainer_test.load_model(args_test.model_dir, epoch=args_test.load_epoch)

In [ ]:
# 8. Start Evaluation on New Classes
print("Starting evaluation on new classes...")
trainer_test.test()